In [ ]:
%%writefile submission.py
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

BOARD = 100.0
CENTER_X, CENTER_Y = 50.0, 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY = 1.5
ROTATION_LIMIT = 50.0
TOTAL_STEPS = 500
HORIZON = 80


def dist(ax, ay, bx, by):
    return math.hypot(ax - bx, ay - by)


def fleet_speed(ships):
    if ships <= 1:
        return 1.0
    ratio = math.log(ships) / math.log(1000.0)
    ratio = max(0.0, min(1.0, ratio))
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)


def segment_hits_sun(x1, y1, x2, y2, safety=SUN_SAFETY):
    r = SUN_R + safety
    dx, dy = x2 - x1, y2 - y1
    fx, fy = x1 - CENTER_X, y1 - CENTER_Y
    a = dx * dx + dy * dy
    if a < 1e-9:
        return dist(x1, y1, CENTER_X, CENTER_Y) < r
    b = 2 * (fx * dx + fy * dy)
    c = fx * fx + fy * fy - r * r
    disc = b * b - 4 * a * c
    if disc < 0:
        return False
    disc = math.sqrt(disc)
    t1 = (-b - disc) / (2 * a)
    t2 = (-b + disc) / (2 * a)
    return (0 <= t1 <= 1) or (0 <= t2 <= 1)


def safe_angle_and_distance(sx, sy, tx, ty):
    direct = dist(sx, sy, tx, ty)
    if not segment_hits_sun(sx, sy, tx, ty):
        return math.atan2(ty - sy, tx - sx), direct
    vx, vy = tx - sx, ty - sy
    norm = math.hypot(vx, vy)
    if norm < 1e-9:
        return math.atan2(ty - sy, tx - sx), direct
    nx, ny = -vy / norm, vx / norm
    best = None
    for sign in (1.0, -1.0):
        for mult in (1.8, 2.3, 3.0, 4.0):
            wx = CENTER_X + sign * nx * (SUN_R * mult)
            wy = CENTER_Y + sign * ny * (SUN_R * mult)
            if segment_hits_sun(sx, sy, wx, wy, safety=SUN_SAFETY) or segment_hits_sun(wx, wy, tx, ty, safety=SUN_SAFETY):
                continue
            d = dist(sx, sy, wx, wy) + dist(wx, wy, tx, ty)
            if best is None or d < best[0]:
                best = (d, wx, wy)
            break
    if best is None:
        return math.atan2(ty - sy, tx - sx), direct * 1.8
    _, wx, wy = best
    return math.atan2(wy - sy, wx - sx), best[0]


def predict_planet_position(planet, initial_by_id, angular_velocity, turns):
    init = initial_by_id.get(planet.id)
    if init is None:
        return planet.x, planet.y
    orbital_r = dist(init.x, init.y, CENTER_X, CENTER_Y)
    if orbital_r + init.radius >= ROTATION_LIMIT:
        return planet.x, planet.y
    cur_ang = math.atan2(planet.y - CENTER_Y, planet.x - CENTER_X)
    new_ang = cur_ang + angular_velocity * turns
    return (CENTER_X + orbital_r * math.cos(new_ang),
            CENTER_Y + orbital_r * math.sin(new_ang))


def predict_comet_position(planet_id, comets, turns):
    for g in comets:
        pids = g.get("planet_ids", [])
        if planet_id not in pids:
            continue
        idx = pids.index(planet_id)
        paths = g.get("paths", [])
        path_index = g.get("path_index", 0)
        if idx >= len(paths):
            return None
        path = paths[idx]
        future_idx = path_index + int(turns)
        if 0 <= future_idx < len(path):
            return path[future_idx][0], path[future_idx][1]
        return None
    return None


def comet_remaining_life(planet_id, comets):
    for g in comets:
        pids = g.get("planet_ids", [])
        if planet_id not in pids:
            continue
        idx = pids.index(planet_id)
        paths = g.get("paths", [])
        path_index = g.get("path_index", 0)
        if idx < len(paths):
            return max(0, len(paths[idx]) - path_index)
    return 0


def travel_time(src_x, src_y, tgt_x, tgt_y, ships):
    _, d = safe_angle_and_distance(src_x, src_y, tgt_x, tgt_y)
    return max(1, int(math.ceil(d / fleet_speed(max(1, ships)))))


def aim_with_prediction(src, target, ships, initial_by_id, ang_vel, comets, comet_ids):
    tx, ty = target.x, target.y
    for _ in range(5):
        angle, d = safe_angle_and_distance(src.x, src.y, tx, ty)
        turns = max(1, int(math.ceil(d / fleet_speed(max(1, ships)))))
        if target.id in comet_ids:
            pos = predict_comet_position(target.id, comets, turns)
            if pos is None:
                return None
            ntx, nty = pos
        else:
            ntx, nty = predict_planet_position(target, initial_by_id, ang_vel, turns)
        if abs(ntx - tx) < 0.3 and abs(nty - ty) < 0.3:
            tx, ty = ntx, nty
            break
        tx, ty = ntx, nty
    angle, d = safe_angle_and_distance(src.x, src.y, tx, ty)
    turns = max(1, int(math.ceil(d / fleet_speed(max(1, ships)))))
    return angle, turns, tx, ty


def fleet_target_planet(fleet, planets):
    best_p, best_t = None, 1e9
    fvx, fvy = math.cos(fleet.angle), math.sin(fleet.angle)
    sp = fleet_speed(fleet.ships)
    for p in planets:
        dx, dy = p.x - fleet.x, p.y - fleet.y
        proj = dx * fvx + dy * fvy
        if proj <= 0:
            continue
        perp = abs(dx * fvy - dy * fvx)
        if perp > p.radius + 1.3:
            continue
        t = proj / sp
        if t < best_t and t <= HORIZON:
            best_t = t
            best_p = p
    if best_p is None:
        return None, None
    return best_p, int(math.ceil(best_t))


def simulate_planet_future(planet, arrivals, player_self, horizon):
    if not arrivals:
        if planet.owner == -1:
            return [(0, planet.owner, planet.ships)]
        return [(horizon, planet.owner, planet.ships + planet.production * horizon)]
    events = sorted(arrivals, key=lambda a: a[0])
    timeline = []
    garrison = planet.ships
    owner = planet.owner
    last_t = 0
    i = 0
    n = len(events)
    while i < n:
        t = events[i][0]
        if t > horizon:
            break
        if owner != -1 and t > last_t:
            garrison += (t - last_t) * planet.production
        group = []
        while i < n and events[i][0] == t:
            group.append(events[i])
            i += 1
        by_owner = {}
        for _, o, s in group:
            by_owner[o] = by_owner.get(o, 0) + s
        if owner in by_owner:
            garrison += by_owner.pop(owner)
        attackers = sorted(by_owner.items(), key=lambda x: -x[1])
        while len(attackers) >= 2 and attackers[0][1] == attackers[1][1]:
            attackers = attackers[2:]
        if attackers:
            top_owner, top_ships = attackers[0]
            second = attackers[1][1] if len(attackers) > 1 else 0
            effective = top_ships - second
            if effective > garrison:
                garrison = effective - garrison
                owner = top_owner
            elif effective > 0:
                garrison -= effective
        last_t = t
        timeline.append((t, owner, max(0, garrison)))
    if last_t < horizon and owner != -1:
        garrison += (horizon - last_t) * planet.production
        timeline.append((horizon, owner, max(0, garrison)))
    return timeline


def net_defense_needed(planet, arrivals, player):
    if not arrivals:
        return 0
    if planet.owner != player:
        return 0
    events = sorted(arrivals, key=lambda a: a[0])
    garrison = planet.ships
    last_t = 0
    deficit = 0
    i = 0
    n = len(events)
    while i < n:
        t = events[i][0]
        garrison += (t - last_t) * planet.production
        group = []
        while i < n and events[i][0] == t:
            group.append(events[i])
            i += 1
        friendly = sum(s for _, o, s in group if o == player)
        enemy = sum(s for _, o, s in group if o != player)
        garrison += friendly - enemy
        if garrison < 0:
            deficit = max(deficit, -garrison)
            garrison = 0
        last_t = t
    return deficit


def safe_ships_available(planet, arrivals, player, horizon):
    if planet.owner != player:
        return 0
    if not arrivals:
        return planet.ships
    events = sorted(arrivals, key=lambda a: a[0])
    garrison = planet.ships
    min_future = garrison
    last_t = 0
    i = 0
    while i < len(events):
        t = events[i][0]
        if t > horizon:
            break
        garrison += (t - last_t) * planet.production
        group = []
        while i < len(events) and events[i][0] == t:
            group.append(events[i])
            i += 1
        friendly = sum(s for _, o, s in group if o == player)
        enemy = sum(s for _, o, s in group if o != player)
        garrison += friendly - enemy
        if garrison < min_future:
            min_future = garrison
        last_t = t
    return max(0, min_future)


def nearest_distance_to_set(px, py, planets_set):
    if not planets_set:
        return 1e9
    return min(dist(px, py, p.x, p.y) for p in planets_set)


def relative_reaction_time(tgt, me_planets, enemy_planets):
    my_time = min((travel_time(p.x, p.y, tgt.x, tgt.y, p.ships or 1) for p in me_planets), default=1e9)
    enemy_time = min((travel_time(p.x, p.y, tgt.x, tgt.y, p.ships or 1) for p in enemy_planets), default=1e9)
    return my_time, enemy_time


def indirect_wealth(planet, all_planets, player):
    w = 0.0
    for p in all_planets:
        if p.id == planet.id:
            continue
        d = dist(planet.x, planet.y, p.x, p.y)
        if d < 1:
            continue
        factor = p.production / (d + 10.0)
        if p.owner == player:
            w += factor * 0.5
        elif p.owner == -1:
            w += factor * 1.0
        else:
            w += factor * 1.3
    return w


def agent(obs):
    if isinstance(obs, dict):
        get = obs.get
    else:
        get = lambda k, d=None: getattr(obs, k, d)

    player = get("player", 0)
    step = get("step", 0) or 0
    raw_planets = get("planets", []) or []
    raw_fleets = get("fleets", []) or []
    ang_vel = get("angular_velocity", 0.0) or 0.0
    raw_init = get("initial_planets", []) or []
    comets = get("comets", []) or []
    comet_ids = set(get("comet_planet_ids", []) or [])

    planets = [Planet(*p) for p in raw_planets]
    fleets = [Fleet(*f) for f in raw_fleets]
    initial_by_id = {Planet(*p).id: Planet(*p) for p in raw_init}
    planet_by_id = {p.id: p for p in planets}

    my_planets = [p for p in planets if p.owner == player]
    if not my_planets:
        return []

    enemy_planets = [p for p in planets if p.owner != player and p.owner != -1]
    neutral_planets = [p for p in planets if p.owner == -1]

    remaining_steps = max(1, TOTAL_STEPS - step)
    is_early = step < 35
    is_mid_early = step < 80
    is_late = remaining_steps < 60
    is_very_late = remaining_steps < 25

    my_total = sum(p.ships for p in my_planets) + sum(int(f.ships) for f in fleets if f.owner == player)
    enemy_total = sum(p.ships for p in enemy_planets) + sum(int(f.ships) for f in fleets if f.owner != player)
    my_prod = sum(p.production for p in my_planets)
    enemy_prod = sum(p.production for p in enemy_planets)

    domination = (my_total - enemy_total) / max(1, my_total + enemy_total)
    finishing = domination > 0.35 and my_prod > enemy_prod * 1.3 and step > 100
    behind = domination < -0.25

    arrivals_by_planet = {p.id: [] for p in planets}
    for f in fleets:
        tgt, eta = fleet_target_planet(f, planets)
        if tgt is None:
            continue
        arrivals_by_planet[tgt.id].append((eta, f.owner, int(f.ships)))

    reserve = {}
    for p in my_planets:
        need = net_defense_needed(p, arrivals_by_planet[p.id], player)
        reserve[p.id] = min(p.ships, need)

    safe_available = {}
    for p in my_planets:
        safe_available[p.id] = safe_ships_available(p, arrivals_by_planet[p.id], player, HORIZON)

    doomed_planets = set()
    for p in my_planets:
        timeline = simulate_planet_future(p, arrivals_by_planet[p.id], player, HORIZON)
        final_owner = timeline[-1][1] if timeline else p.owner
        will_be_lost = any(e[1] != player and e[1] != -1 for e in timeline)
        if will_be_lost and reserve[p.id] >= p.ships:
            doomed_planets.add(p.id)

    available = {}
    for p in my_planets:
        if p.id in doomed_planets:
            available[p.id] = p.ships
        elif finishing:
            available[p.id] = max(0, p.ships - max(0, reserve[p.id] - 5))
        elif is_late:
            available[p.id] = max(0, p.ships - reserve[p.id])
        else:
            available[p.id] = min(safe_available[p.id], max(0, p.ships - reserve[p.id]))

    inbound_friendly = {}
    inbound_enemy = {}
    inbound_friendly_eta = {}
    for f in fleets:
        tgt, eta = fleet_target_planet(f, planets)
        if tgt is None:
            continue
        if f.owner == player:
            inbound_friendly[tgt.id] = inbound_friendly.get(tgt.id, 0) + int(f.ships)
            if tgt.id not in inbound_friendly_eta or inbound_friendly_eta[tgt.id] > eta:
                inbound_friendly_eta[tgt.id] = eta
        else:
            inbound_enemy[tgt.id] = inbound_enemy.get(tgt.id, 0) + int(f.ships)

    moves = []

    def compute_defender(tgt, arrival_turns):
        if tgt.owner == -1:
            defender = tgt.ships
        elif tgt.owner == player:
            arrs = arrivals_by_planet.get(tgt.id, [])
            enemy_before = sum(s for t, o, s in arrs if o != player and t <= arrival_turns)
            friend_before = sum(s for t, o, s in arrs if o == player and t < arrival_turns)
            defender = max(0, tgt.ships + tgt.production * arrival_turns + friend_before - enemy_before)
        else:
            arrs = arrivals_by_planet.get(tgt.id, [])
            enemy_same_owner_reinforce = sum(s for t, o, s in arrs if o == tgt.owner and t <= arrival_turns)
            friend_before = sum(s for t, o, s in arrs if o == player and t < arrival_turns)
            defender = tgt.ships + tgt.production * arrival_turns + enemy_same_owner_reinforce - friend_before
        return max(0, defender)

    def is_safe_neutral(tgt):
        if tgt.owner != -1:
            return False
        my_t, en_t = relative_reaction_time(tgt, my_planets, enemy_planets)
        return my_t <= en_t - 2

    def is_contested_neutral(tgt):
        if tgt.owner != -1:
            return False
        my_t, en_t = relative_reaction_time(tgt, my_planets, enemy_planets)
        return abs(my_t - en_t) <= 2

    def target_value(tgt, arrival_turns, cost):
        prod = tgt.production
        turns_profit = max(1, remaining_steps - arrival_turns)
        if tgt.id in comet_ids:
            life = comet_remaining_life(tgt.id, comets)
            turns_profit = max(0, min(turns_profit, life - arrival_turns))
            if turns_profit <= 0:
                return -1
        base = prod * turns_profit
        iw = indirect_wealth(tgt, planets, player)
        value = base + iw * turns_profit * 0.15
        if tgt.owner != player and tgt.owner != -1:
            value *= 1.9
            if finishing:
                value *= 1.3
        if tgt.owner == -1:
            if is_safe_neutral(tgt):
                value *= 1.2
            elif is_contested_neutral(tgt):
                value *= 0.75
            if is_early:
                value *= 1.35
        return value

    def compute_needed(tgt, arrival_turns):
        defender = compute_defender(tgt, arrival_turns)
        already_sending = inbound_friendly.get(tgt.id, 0)
        needed = defender + 1 - already_sending
        return max(1, needed)

    candidates = []
    for src in my_planets:
        if available[src.id] <= 0:
            continue
        src_ships = available[src.id]
        for tgt in planets:
            if tgt.id == src.id or tgt.owner == player:
                continue
            d0 = dist(src.x, src.y, tgt.x, tgt.y)
            if d0 > 145:
                continue
            est_turns = travel_time(src.x, src.y, tgt.x, tgt.y, max(10, src_ships))
            if is_very_late and est_turns > remaining_steps - 3:
                continue
            if tgt.id in comet_ids and est_turns >= comet_remaining_life(tgt.id, comets):
                continue

            est_needed = compute_needed(tgt, est_turns)
            if est_needed > src_ships:
                continue

            aim = aim_with_prediction(src, tgt, est_needed, initial_by_id, ang_vel, comets, comet_ids)
            if aim is None:
                continue
            angle, turns, _, _ = aim

            ships_needed = compute_needed(tgt, turns)
            if ships_needed > src_ships:
                continue

            if segment_hits_sun(src.x, src.y,
                                src.x + math.cos(angle) * 3,
                                src.y + math.sin(angle) * 3, safety=0.4):
                continue

            if tgt.owner == -1 and is_contested_neutral(tgt):
                my_t, en_t = relative_reaction_time(tgt, my_planets, enemy_planets)
                if en_t <= turns:
                    ships_needed = min(src_ships, int(ships_needed * 1.4) + 3)

            cost = ships_needed + turns * 0.5
            value = target_value(tgt, turns, cost)
            if value <= 0:
                continue
            score = value / (cost + 1.0)

            if tgt.owner != player and tgt.owner != -1:
                if finishing:
                    score *= 1.4
                if behind:
                    score *= 0.9
            if is_early and tgt.owner == -1 and ships_needed <= 18:
                score *= 1.6
            if tgt.id in comet_ids and tgt.production >= 1:
                score *= 1.1

            candidates.append((score, src.id, tgt.id, angle, ships_needed, turns, value))

    candidates.sort(key=lambda x: -x[0])

    target_commitments = {}
    planet_dispatched = {}
    for score, sid, tid, angle, ships, turns, value in candidates:
        already = target_commitments.get(tid, 0)
        tgt = planet_by_id[tid]
        current_eta = turns
        base_needed = compute_defender(tgt, current_eta) + 1
        missing = max(0, base_needed - inbound_friendly.get(tid, 0) - already)
        if missing <= 0:
            continue
        src_avail = available[sid] - planet_dispatched.get(sid, 0)
        if src_avail <= 0:
            continue
        send = min(src_avail, missing)
        if send < 1:
            continue
        moves.append([sid, float(angle), int(send)])
        planet_dispatched[sid] = planet_dispatched.get(sid, 0) + send
        target_commitments[tid] = already + send

    for sid in list(planet_dispatched.keys()):
        available[sid] = max(0, available[sid] - planet_dispatched[sid])

    if not is_very_late:
        for src in my_planets:
            if available[src.id] < 8:
                continue
            src_ships = available[src.id]
            best = None
            for tgt in planets:
                if tgt.id == src.id or tgt.owner == player:
                    continue
                d0 = dist(src.x, src.y, tgt.x, tgt.y)
                if d0 > 140:
                    continue
                est_turns = travel_time(src.x, src.y, tgt.x, tgt.y, max(10, src_ships))
                if is_late and est_turns > remaining_steps - 5:
                    continue
                if tgt.id in comet_ids and est_turns >= comet_remaining_life(tgt.id, comets):
                    continue
                defender = compute_defender(tgt, est_turns)
                committed = target_commitments.get(tgt.id, 0) + inbound_friendly.get(tgt.id, 0)
                missing = max(0, defender + 1 - committed)
                if missing <= 0:
                    continue
                if missing > src_ships and committed == 0:
                    continue
                send = min(src_ships, missing)
                if send < 5:
                    continue
                if committed + send < defender + 1:
                    continue
                value = target_value(tgt, est_turns, send)
                if value <= 0:
                    continue
                score = value / (send + est_turns * 0.5 + 1.0)
                if best is None or score > best[0]:
                    best = (score, tgt, send, est_turns)
            if best is None:
                continue
            score, tgt, send, est_turns = best
            aim = aim_with_prediction(src, tgt, send, initial_by_id, ang_vel, comets, comet_ids)
            if aim is None:
                continue
            angle, turns, _, _ = aim
            if segment_hits_sun(src.x, src.y,
                                src.x + math.cos(angle) * 3,
                                src.y + math.sin(angle) * 3, safety=0.4):
                continue
            moves.append([src.id, float(angle), int(send)])
            available[src.id] -= send
            target_commitments[tgt.id] = target_commitments.get(tgt.id, 0) + send

    for sid in doomed_planets:
        src = planet_by_id[sid]
        if available[sid] <= 0:
            continue
        best = None
        for tgt in enemy_planets:
            d0 = dist(src.x, src.y, tgt.x, tgt.y)
            if d0 > 100:
                continue
            turns_est = travel_time(src.x, src.y, tgt.x, tgt.y, available[sid])
            if turns_est >= remaining_steps - 2:
                continue
            val = tgt.production * max(1, remaining_steps - turns_est) * 0.8
            score = val / (turns_est + 1.0)
            if best is None or score > best[0]:
                best = (score, tgt, turns_est)
        if best is None:
            for tgt in my_planets:
                if tgt.id == src.id or tgt.id in doomed_planets:
                    continue
                d0 = dist(src.x, src.y, tgt.x, tgt.y)
                if d0 > 80:
                    continue
                turns_est = travel_time(src.x, src.y, tgt.x, tgt.y, available[sid])
                score = tgt.production / (turns_est + 1.0)
                if best is None or score > best[0]:
                    best = (score, tgt, turns_est)
        if best is None:
            continue
        _, tgt, _ = best
        aim = aim_with_prediction(src, tgt, available[sid], initial_by_id, ang_vel, comets, comet_ids)
        if aim is None:
            continue
        angle, turns, _, _ = aim
        if segment_hits_sun(src.x, src.y,
                            src.x + math.cos(angle) * 3,
                            src.y + math.sin(angle) * 3, safety=0.4):
            continue
        send = available[sid]
        if send >= 1:
            moves.append([sid, float(angle), int(send)])
            available[sid] = 0

    if not is_very_late and (enemy_planets or neutral_planets) and len(my_planets) > 1:
        ref_set = enemy_planets if enemy_planets else neutral_planets
        front_dist = {p.id: nearest_distance_to_set(p.x, p.y, ref_set) for p in my_planets}
        sorted_by_rear = sorted(my_planets, key=lambda p: -front_dist[p.id])
        front = min(my_planets, key=lambda p: front_dist[p.id])
        for r in sorted_by_rear:
            if r.id == front.id:
                continue
            if r.id in doomed_planets:
                continue
            if front_dist[r.id] < front_dist[front.id] * 1.2:
                continue
            if available[r.id] < 15:
                continue
            mid_candidates = [p for p in my_planets
                              if p.id != r.id and p.id not in doomed_planets
                              and front_dist[p.id] < front_dist[r.id] * 0.75]
            forward_target = None
            if mid_candidates:
                mid_candidates.sort(key=lambda p: dist(r.x, r.y, p.x, p.y))
                forward_target = mid_candidates[0]
            else:
                forward_target = front
            if forward_target.id == r.id:
                continue
            send_ratio = 0.8 if finishing else 0.65
            send = int(available[r.id] * send_ratio)
            if send < 10:
                continue
            aim = aim_with_prediction(r, forward_target, send, initial_by_id, ang_vel, comets, comet_ids)
            if aim is None:
                continue
            angle, turns, _, _ = aim
            if turns > 40:
                continue
            if segment_hits_sun(r.x, r.y,
                                r.x + math.cos(angle) * 3,
                                r.y + math.sin(angle) * 3, safety=0.4):
                continue
            moves.append([r.id, float(angle), int(send)])
            available[r.id] -= send

    if finishing and enemy_planets:
        weak_enemy = sorted(enemy_planets, key=lambda p: p.ships + p.production * 10)
        for src in my_planets:
            if available[src.id] < 25:
                continue
            for tgt in weak_enemy[:3]:
                if tgt.id in target_commitments:
                    continue
                d0 = dist(src.x, src.y, tgt.x, tgt.y)
                if d0 > 120:
                    continue
                turns_est = travel_time(src.x, src.y, tgt.x, tgt.y, available[src.id])
                if turns_est > remaining_steps - 5:
                    continue
                needed = compute_defender(tgt, turns_est) + 1
                committed = target_commitments.get(tgt.id, 0) + inbound_friendly.get(tgt.id, 0)
                missing = max(0, needed - committed)
                if missing <= 0:
                    continue
                send = min(available[src.id], missing + 5)
                if send < 10:
                    continue
                aim = aim_with_prediction(src, tgt, send, initial_by_id, ang_vel, comets, comet_ids)
                if aim is None:
                    continue
                angle, turns, _, _ = aim
                if segment_hits_sun(src.x, src.y,
                                    src.x + math.cos(angle) * 3,
                                    src.y + math.sin(angle) * 3, safety=0.4):
                    continue
                moves.append([src.id, float(angle), int(send)])
                available[src.id] -= send
                target_commitments[tgt.id] = target_commitments.get(tgt.id, 0) + send
                break

    dedup = {}
    for sid, ang, sh in moves:
        key = (sid, round(ang, 4))
        if key in dedup:
            dedup[key] = (sid, ang, dedup[key][2] + sh)
        else:
            dedup[key] = (sid, ang, sh)
    final_moves = []
    used_per_src = {}
    for sid, ang, sh in dedup.values():
        src = planet_by_id[sid]
        max_allowed = src.ships - used_per_src.get(sid, 0)
        send = min(sh, max_allowed)
        if send >= 1:
            final_moves.append([sid, float(ang), int(send)])
            used_per_src[sid] = used_per_src.get(sid, 0) + send

    return final_moves